In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import gc

In [2]:
df = pd.read_csv('features/features_one_week.csv', sep=';')
df = df[df['ОстатокНачалоНедели'] % 1 == 0]
df = df[df['Количество'] >= 0]
df = df[df['Количество'] <= 350]
df['НеделяНачало'] = pd.to_datetime(df['НеделяНачало'])
df.shape

(2471151, 20)

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
df_unique = df.drop_duplicates(subset='Номенклатура').copy()
tfidf = TfidfVectorizer()
tfidf.fit(df_unique['Номенклатура'])
text_new = []
for i in range(len(df_unique)):
    s = df_unique['Номенклатура'].iloc[i]
    df_1 = pd.DataFrame(tfidf.transform([s]).T.todense())
    df_1 = df_1[df_1.values > 0]
    text_new.append(df_1.mean().iloc[0])
df_unique['tfidf_mean'] = pd.Series(text_new, index=df_unique.index)
df = df.merge(df_unique[['Номенклатура', 'tfidf_mean']], on='Номенклатура', how='left')
del df_unique

In [ ]:
nom_sup = pd.read_csv('features/nomenclature_with_supplier.csv', sep=',')
nom_sup.loc[(nom_sup['supplier'] == 'НК') | nom_sup['supplier'].isna(), 'supplier'] = 'Нет значения'
df = df.merge(nom_sup, how='left', left_on='Номенклатура', right_on='product')
df = df.drop(columns=['product'])
df['supplier'] = df['supplier'].fillna('Нет значения')
del nom_sup

s = pd.to_datetime(df['ДатаПоследнегоПоступления'], errors='coerce')
df['МесяцПоследнегоПоступления'] = s.dt.month
df['ГодПоследнегоПоступления']   = s.dt.year
df.drop(columns=['ДатаПоследнегоПоступления'], inplace=True)

In [5]:
# Агрегация недель
df['МесяцНачало'] = df['НеделяНачало'].dt.to_period('M').dt.start_time

df_monthly = df.groupby(['КодТовара', 'МесяцНачало']).agg({
    'Номенклатура':               'last',
    'Папка1':                     'last',
    'Папка2':                     'last',
    'Поставщик':                  'last',
    'ЕдиницаИзмерения':           'last',
    'ТоварнаяКатегория':          'last',
    'Количество':                 'sum',
    'Розничная30%':               'last',
    'ЗакупочнаяЦена':             'last',
    'ОстатокНачалоНедели':        'first',
    'temp_mean_week':             'mean',
    'precip_sum_week':            'sum',
    'temp_max_week':              'max',
    'temp_min_week':              'min',
    'НДС':                        'last',
    'days_off_in_week':           'sum',
    'tfidf_mean':                 'last',
    'supplier':                   'last',
    'МесяцПоследнегоПоступления': 'last',
    'ГодПоследнегоПоступления':   'last',
}).reset_index()

df_monthly = df_monthly.rename(columns={
    'ОстатокНачалоНедели': 'ОстатокНачалоМесяца',
    'temp_mean_week':  'temp_mean_month',
    'precip_sum_week': 'precip_sum_month',
    'temp_max_week':   'temp_max_month',
    'temp_min_week':   'temp_min_month',
    'days_off_in_week':'days_off_month',
})

print(f'Строк: {len(df_monthly)}, уникальных месяцев: {df_monthly["МесяцНачало"].nunique()}')
del df; gc.collect()

Строк: 623830, уникальных месяцев: 103


47

In [6]:
# Матрица месячных продаж
df_monthly['КодТовара']   = df_monthly['КодТовара'].astype(str).str.strip().str.replace(r'\.0$', '', regex=True)
df_monthly['МесяцНачало'] = pd.to_datetime(df_monthly['МесяцНачало']).dt.tz_localize(None).dt.normalize()

df_sells = (
    df_monthly.set_index(['КодТовара', 'МесяцНачало'])[['Количество']]
              .unstack(level=-1).fillna(0)
              .sort_index().sort_index(axis=1)
)
df_sells.columns = df_sells.columns.get_level_values(1)
df_sells.columns = pd.to_datetime(df_sells.columns, errors='coerce').tz_localize(None).normalize()
df_sells.index = df_sells.index.astype(str).str.strip().str.replace(r'\.0$', '', regex=True)
df_sells.shape

(35442, 103)

In [7]:
from datetime import timedelta

def get_timespan_m(df, dt, minus, periods):
    cols = pd.date_range(
        pd.to_datetime(dt) - pd.DateOffset(months=minus),
        periods=periods, freq='MS'
    )
    return df.reindex(columns=cols, fill_value=0)


def prepare_month(df, t):
    X = {}
    # Лаговые фичи по окнам [2, 4, 6, 12] месяцев
    for i in [2, 4, 6, 12]:
        tmp = get_timespan_m(df, t, i, i)
        X[f'diff_{i}_mean']  = tmp.diff(axis=1).mean(axis=1).values
        X[f'mean_{i}_decay'] = (tmp * np.power(0.9, np.arange(i)[::-1])).sum(axis=1).values
        X[f'mean_{i}']       = tmp.mean(axis=1).values
        X[f'median_{i}']     = tmp.median(axis=1).values
        X[f'min_{i}']        = tmp.min(axis=1).values
        X[f'max_{i}']        = tmp.max(axis=1).values
        X[f'std_{i}']        = tmp.std(axis=1).values

    for i in [2, 4, 6, 12]:
        tmp = get_timespan_m(df, pd.to_datetime(t) - pd.DateOffset(months=1), i, i)
        X[f'diff_{i}_mean_2']  = tmp.diff(axis=1).mean(axis=1).values
        X[f'mean_{i}_decay_2'] = (tmp * np.power(0.9, np.arange(i)[::-1])).sum(axis=1).values
        X[f'mean_{i}_2']       = tmp.mean(axis=1).values
        X[f'median_{i}_2']     = tmp.median(axis=1).values
        X[f'min_{i}_2']        = tmp.min(axis=1).values
        X[f'max_{i}_2']        = tmp.max(axis=1).values
        X[f'std_{i}_2']        = tmp.std(axis=1).values
    return pd.DataFrame(X)

In [8]:
# Лаговые фичи
months = df_monthly['МесяцНачало'].dropna().drop_duplicates().sort_values().tolist()
print(f'Всего месяцев: {len(months)}')

out_frames = []
for m in months:
    feats = prepare_month(df_sells, m)
    if isinstance(feats.index, pd.RangeIndex):
        feats.index = df_sells.index
    feats.index.name = 'КодТовара'
    feats = feats.reset_index()
    feats['КодТовара']   = feats['КодТовара'].astype(str).str.strip().str.replace(r'\.0$', '', regex=True)
    feats['МесяцНачало'] = m

    df_m = df_monthly[df_monthly['МесяцНачало'] == m].copy()
    if df_m.empty:
        continue
    df_m = df_m.merge(feats, on=['КодТовара', 'МесяцНачало'], how='left')
    out_frames.append(df_m)
    del feats, df_m

df = pd.concat(out_frames, ignore_index=True)
del out_frames, df_monthly, df_sells; gc.collect()
print(f'Итого строк: {len(df)}')

Всего месяцев: 103
Итого строк: 623830


In [9]:
# Фильтрация
df = df[(df['Папка1'] != 'разобрать')]
df = df[df['ЕдиницаИзмерения'] == 'шт'].drop('ЕдиницаИзмерения', axis=1)
df = df[df['ТоварнаяКатегория'] == 'Штучный товар'].drop('ТоварнаяКатегория', axis=1)
# Проверяем оба столбца на дробные значения
bad_cols = ['ОстатокНачалоМесяца', 'Количество']
bad_codes = df.loc[(df[bad_cols].astype(float) % 1 != 0).any(axis=1), 'КодТовара'].unique()
df = df[~df['КодТовара'].isin(bad_codes)]
df = df[df['Номенклатура'] != '1']

df = df[df['МесяцНачало'] > '2018-01-01']
df = df[df['Количество'] >= 0]
df = df[df['Количество'] <= 75]
df = df.sort_values('МесяцНачало', ascending=True, kind='mergesort')
print(f'После фильтрации: {len(df)}')

После фильтрации: 595054


In [10]:
# Train/Test split
X = df.drop('Количество', axis=1).copy()
y = df['Количество'].copy()

X['МесяцНачало'] = pd.to_datetime(X['МесяцНачало'])
X_test  = X[X.МесяцНачало >= '2025-08-01'].copy()
X_train = X[X.МесяцНачало <  '2025-08-01'].copy()
y_test  = y[y.index.isin(X_test.index)]
y_train = y[y.index.isin(X_train.index)]

for frame in [X_test, X_train, X]:
    frame['ТекущийМесяц'] = pd.to_datetime(frame['МесяцНачало'], errors='coerce').dt.month
    frame['ТекущийГод']   = pd.to_datetime(frame['МесяцНачало'], errors='coerce').dt.year
    frame.drop(columns=['МесяцНачало'], inplace=True)

object_cols = ['Папка1', 'Папка2', 'Поставщик', 'supplier',
               'МесяцПоследнегоПоступления', 'ГодПоследнегоПоступления',
               'ТекущийМесяц', 'ТекущийГод']
for frame in [X, X_test, X_train]:
    frame[object_cols] = frame[object_cols].astype(object)

print(f'Train: {len(X_train)}, Test: {len(X_test)}')

Train: 558269, Test: 36785


In [11]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.compose import ColumnTransformer
from category_encoders import TargetEncoder
from category_encoders.one_hot import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import make_scorer
from xgboost import XGBRegressor


# WAPE
def wape(y_true, y_pred):
    """WAPE = sum(|y_true - y_pred|) / sum(|y_true|)."""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom  = np.sum(np.abs(y_true))
    if denom == 0:
        return 0.0
    return np.sum(np.abs(y_true - y_pred)) / denom


wape_scorer = make_scorer(wape, greater_is_better=False)


cols_for_ohe = [x for x in object_cols if X_train[x].nunique() < 5]
cols_for_mte = [x for x in object_cols if X_train[x].nunique() >= 5]
numeric_cols  = list(X_train.select_dtypes(exclude='object').columns)

cols_for_ohe_idx = [list(X_train.columns).index(c) for c in cols_for_ohe]
cols_for_mte_idx = [list(X_train.columns).index(c) for c in cols_for_mte]
numeric_cols_idx = [list(X_train.columns).index(c) for c in numeric_cols]

col_transform = ColumnTransformer([
    ('OHE', OneHotEncoder(),  cols_for_ohe_idx),
    ('MTE', TargetEncoder(),  cols_for_mte_idx),
    ('SC',  StandardScaler(), numeric_cols_idx),
])
col_transform.fit(X_train, y_train)

pipe = Pipeline([
    ('column_transformer', col_transform),
    ('gradient_boosting', XGBRegressor(
        objective='reg:absoluteerror',
        random_state=42,
        n_estimators=300,
        subsample=1.0,
        min_child_weight=1,
        max_depth=6,
        learning_rate=0.1,
        gamma=1
    ))
])
pipe.fit(X_train, y_train)

train_preds = pipe.predict(X_train)
test_preds  = pipe.predict(X_test)
print('Без CV (гиперпараметры подобраны из головы)')
print(f'WAPE train : {wape(y_train, train_preds):.4f}, WAPE test: {wape(y_test, test_preds):.4f}')

Без CV (гиперпараметры подобраны из головы)
WAPE train : 0.5823, WAPE test: 0.8398


In [12]:
# GridSearchCV (CV-метрика — WAPE, 4 комбо x 3 фолда = 12 фитов)
from sklearn.model_selection import GridSearchCV

param_grid = {
    'gradient_boosting__n_estimators': [300, 600],
    'gradient_boosting__max_depth':    [4, 6],
}

cv_fast = TimeSeriesSplit(n_splits=3)

search = GridSearchCV(pipe, param_grid, cv=cv_fast,
                      scoring=wape_scorer,
                      n_jobs=1, verbose=3)
search.fit(X_train, y_train)

print(f'Best params (CV WAPE={-search.best_score_:.5f}):')
print(search.best_params_)

best_model = search.best_estimator_
y_pred = best_model.predict(X_test)
print(f'WAPE лучшей модели на тесте: {wape(y_test, y_pred):.5f}')

Fitting 3 folds for each of 4 candidates, totalling 12 fits
[CV 1/3] END gradient_boosting__max_depth=4, gradient_boosting__n_estimators=300;, score=-0.654 total time=  12.8s
[CV 2/3] END gradient_boosting__max_depth=4, gradient_boosting__n_estimators=300;, score=-0.726 total time=  27.4s
[CV 3/3] END gradient_boosting__max_depth=4, gradient_boosting__n_estimators=300;, score=-0.702 total time=  36.9s
[CV 1/3] END gradient_boosting__max_depth=4, gradient_boosting__n_estimators=600;, score=-0.654 total time=  24.9s
[CV 2/3] END gradient_boosting__max_depth=4, gradient_boosting__n_estimators=600;, score=-0.738 total time=  53.3s
[CV 3/3] END gradient_boosting__max_depth=4, gradient_boosting__n_estimators=600;, score=-0.703 total time= 1.2min
[CV 1/3] END gradient_boosting__max_depth=6, gradient_boosting__n_estimators=300;, score=-0.669 total time=  11.9s
[CV 2/3] END gradient_boosting__max_depth=6, gradient_boosting__n_estimators=300;, score=-0.767 total time=  27.5s
[CV 3/3] END gradien

In [13]:
cv_results = pd.DataFrame(search.cv_results_)

fold_cols = [c for c in cv_results.columns
             if c.startswith('split') and c.endswith('_test_score')]
keep_cols = (['param_gradient_boosting__n_estimators',
              'param_gradient_boosting__max_depth']
             + fold_cols
             + ['mean_test_score', 'std_test_score', 'rank_test_score'])

df_cv = cv_results[keep_cols].copy()

for c in fold_cols + ['mean_test_score']:
    df_cv[c] = -df_cv[c]

df_cv = df_cv.rename(columns={
    'param_gradient_boosting__n_estimators': 'n_estimators',
    'param_gradient_boosting__max_depth':    'max_depth',
    'mean_test_score': 'WAPE_mean',
    'std_test_score':  'WAPE_std',
    'rank_test_score': 'rank',
    **{c: f'fold{i+1}_WAPE' for i, c in enumerate(fold_cols)},
})
df_cv = df_cv.sort_values('rank').reset_index(drop=True).round(5)

print("CV-результаты по фолдам (TimeSeriesSplit, n_splits=3):")
print(df_cv.to_string(index=False))
df_cv

CV-результаты по фолдам (TimeSeriesSplit, n_splits=3):
 n_estimators  max_depth  fold1_WAPE  fold2_WAPE  fold3_WAPE  WAPE_mean  WAPE_std  rank
          300          4     0.65440     0.72640     0.70156    0.69412   0.02986     1
          600          4     0.65450     0.73752     0.70281    0.69828   0.03405     2
          300          6     0.66896     0.76699     0.69233    0.70943   0.04181     3
          600          6     0.67335     0.80248     0.69382    0.72321   0.05667     4


,n_estimators,max_depth,fold1_WAPE,fold2_WAPE,fold3_WAPE,WAPE_mean,WAPE_std,rank
0,300,4,0.65440,0.72640,0.70156,0.69412,0.02986,1
1,600,4,0.65450,0.73752,0.70281,0.69828,0.03405,2
2,300,6,0.66896,0.76699,0.69233,0.70943,0.04181,3
3,600,6,0.67335,0.80248,0.69382,0.72321,0.05667,4


In [14]:
# Финальное качество на train и test
final_model = best_model

train_preds_final = final_model.predict(X_train)
test_preds_final  = final_model.predict(X_test)

wape_train_final = wape(y_train, train_preds_final)
wape_test_final  = wape(y_test,  test_preds_final)

print('=' * 60)
print('ФИНАЛЬНОЕ КАЧЕСТВО XGBoost (месяц)')
print('=' * 60)
print(f'  WAPE train: {wape_train_final:.4f}  (n={len(y_train):,})')
print(f'  WAPE test:  {wape_test_final:.4f}  (n={len(y_test):,})')


ФИНАЛЬНОЕ КАЧЕСТВО XGBoost (месяц)
  WAPE train: 0.6283  (n=558,269)
  WAPE test:  0.8344  (n=36,785)


In [15]:
# Проверка адекватности модели: R² на train и test
#   R² = 1  — идеальное предсказание;
#   R² = 0  — модель не лучше предсказания среднего;
#   R² < 0  — модель хуже тривиального прогноза.
from sklearn.metrics import r2_score

r2_train = r2_score(y_train, final_model.predict(X_train))
r2_test  = r2_score(y_test,  final_model.predict(X_test))

print(f"R² train: {r2_train:.4f}")
print(f"R² test:  {r2_test:.4f}")

R² train: 0.5261
R² test:  0.5555


In [ ]:
import joblib

save_path = 'models/month.pkl'
joblib.dump(best_model, save_path)
print(f'Модель сохранена: {save_path}')

Модель сохранена: D:\learning_projects\uir_all\models\month.pkl
